# Qwen3.8-Flash-Next (G4 / 800K / Vision)

Runs `SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF` (IQ3_S, 819200 context with YaRN 3.125x, BF16 vision projector, `--load-mode none --lazy-mode off`) on a Colab G4 (RTX PRO 6000 Blackwell sm_120) with `llama-server --api-key-file` authentication.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "qwen-flash")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8000"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- parallel llama.cpp sm_120 build/cache restore + 84 GB GGUF + BF16 mmproj staging
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import hashlib, platform, tarfile

CFG = {
    "repo": os.environ.get("CT_QWEN_REPO", "SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF"),
    "quant": os.environ.get("CT_QWEN_QUANT", "IQ3_S"),
    "alias": os.environ.get("CT_QWEN_ALIAS", "qwen3.8-flash-next"),
    "context": int(os.environ.get("CT_QWEN_CONTEXT", "819200")),
    "rope_scaling": os.environ.get("CT_QWEN_ROPE_SCALING", "yarn"),
    "rope_scale": float(os.environ.get("CT_QWEN_ROPE_SCALE", "3.125")),
    "yarn_orig_ctx": int(os.environ.get("CT_QWEN_YARN_ORIG_CTX", "262144")),
    "cache_type": os.environ.get("CT_QWEN_CACHE_TYPE", "f16"),
    "batch": int(os.environ.get("CT_QWEN_BATCH", "512")),
    "ubatch": int(os.environ.get("CT_QWEN_UBATCH", "128")),
    "llama_repo": os.environ.get("CT_LLAMA_REPO", "https://github.com/ggml-org/llama.cpp.git"),
    "llama_commit": os.environ.get("CT_LLAMA_COMMIT", "a0e6dc7384edb8f5d927155e1d26c10b4a310e6d"),
    "service_dir": os.environ.get("CT_QWEN_SERVICE_DIR", "/content/drive/MyDrive/colab-services/qwen-flash"),
    "model_root": os.environ.get("CT_QWEN_MODEL_ROOT", "/content/drive/MyDrive/models"),
    "work_dir": os.environ.get("CT_QWEN_WORK_DIR", "/content/qwen-flash"),
    "vision_projector": {
        "name": "mmproj-Qwen3.8-Flash-Next-BF16.gguf",
        "revision": "eec4e10a2c29b440a2ae85591fa11569ce752963",
        "size": 907542944,
        "sha256": "2e788f8c511d8093c7b43cb87b2fd7e14228340318057f8fb20c86df2efe2355",
    },
}

DEFAULT_MODEL_LOCK = {
    "repo": "SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF",
    "revision": "eec4e10a2c29b440a2ae85591fa11569ce752963",
    "files": [
        {
            "name": "IQ3_S/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-IQ3_S-00001-of-00002.gguf",
            "size": 55063448064,
            "sha256": "40f369eca301ec6ea318dbb671b3132c26a2f0b89aa5458a7051b33215ee296c",
        },
        {
            "name": "IQ3_S/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-IQ3_S-00002-of-00002.gguf",
            "size": 28800138432,
            "sha256": "316b46f3a2dbd68c900f43136ab9449f9dcc3725dfd8c794847c204bc161e113",
        },
    ],
}

def qlog(msg):
    print(time.strftime("[%F %T] ") + msg, flush=True)

def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f"{path.name}.{SERVICE}.{os.getpid()}.tmp")
    tmp.write_text(json.dumps(value, indent=2) + "\n")
    tmp.replace(path)

def cmd_out(command, **kwargs):
    return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT, **kwargs).strip()

def run_logged(command, filename, env=None):
    with open(filename, "a") as handle:
        res = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT, env=env)
    if res.returncode:
        raise RuntimeError(f"Command failed ({res.returncode}); see {filename}")

def digest_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(16 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def require_space(path, needed):
    free = shutil.disk_usage(path).free
    if free < needed:
        raise RuntimeError(f"{path}: need {needed/1e9:.1f} GB, free {free/1e9:.1f} GB")

def restore_valid_bin(bin_dir, env):
    server = bin_dir / "llama-server"
    if not server.is_file():
        return False
    try:
        server.chmod(0o755)
        cmd_out([str(server), "--version"], env=env)
        devices = cmd_out([str(server), "--list-devices"], env=env)
        return "CUDA" in devices and "RTX PRO 6000" in devices
    except (OSError, subprocess.CalledProcessError):
        return False

def prepare_runtime(config, service, work):
    commit_file = service / "llama-commit.txt"
    if not commit_file.exists():
        commit = config.get("llama_commit") or cmd_out(["git", "ls-remote", config["llama_repo"], "HEAD"]).split()[0]
        commit_file.write_text(commit + "\n")
    commit = commit_file.read_text().strip()
    gpu = cmd_out(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"]).splitlines()[0]
    if "RTX PRO 6000" not in gpu or "12.0" not in gpu:
        raise RuntimeError("This deployment requires G4 / RTX PRO 6000 sm_120: " + gpu)
    signature = {
        "llama_commit": commit, "gpu": gpu, "cuda": cmd_out(["nvcc", "--version"]),
        "os_release": Path("/etc/os-release").read_text(), "arch": platform.machine(),
        "glibc": platform.libc_ver(), "cuda_arch": "120", "ggml_native": False,
    }
    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:20]
    cache = Path(config["model_root"]) / ".runtimes" / "llama.cpp" / ("sm120-" + key)
    cache.mkdir(parents=True, exist_ok=True)
    bin_dir = work / "runtime" / "bin"
    bin_dir.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, LD_LIBRARY_PATH=str(bin_dir) + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
    local_manifest = work / "runtime-manifest.json"
    if bin_dir.exists() and any(bin_dir.iterdir()) and (
        not local_manifest.exists() or json.loads(local_manifest.read_text()) != json.loads(json.dumps(signature))
    ):
        shutil.rmtree(bin_dir)
        bin_dir.mkdir()
    archive = cache / "runtime.tar"
    checksum = cache / "runtime.sha256"
    if not restore_valid_bin(bin_dir, env) and archive.exists() and checksum.exists():
        qlog("Restoring compatible build cache from Drive: " + str(cache))
        local_tar = work / "runtime.tar"
        shutil.copyfile(archive, local_tar)
        if digest_file(local_tar) != checksum.read_text().strip():
            raise RuntimeError("Build cache checksum mismatch; refusing corrupt archive")
        with tarfile.open(local_tar) as bundle:
            bundle.extractall(work / "runtime", filter="data")
        local_tar.unlink()
    if not restore_valid_bin(bin_dir, env):
        qlog("Building pinned llama.cpp for CUDA sm_120; progress: " + str(work / "build.log"))
        source = work / "llama.cpp"
        if not (source / ".git").exists():
            run_logged(["git", "clone", "--no-checkout", "--filter=blob:none", config["llama_repo"], str(source)], work / "build.log")
        run_logged(["git", "-C", str(source), "fetch", "--depth", "1", "origin", commit], work / "build.log")
        run_logged(["git", "-C", str(source), "checkout", "--detach", commit], work / "build.log")
        run_logged(["cmake", "-S", str(source), "-B", str(source / "build"),
                    "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=120",
                    "-DGGML_NATIVE=OFF", "-DCMAKE_BUILD_TYPE=Release",
                    "-DLLAMA_CURL=OFF", "-DLLAMA_OPENSSL=OFF", "-DLLAMA_BUILD_TESTS=OFF"], work / "build.log")
        run_logged(["cmake", "--build", str(source / "build"), "--config", "Release",
                    "-j", str(min(12, os.cpu_count() or 4))], work / "build.log")
        shutil.copytree(source / "build" / "bin", bin_dir, dirs_exist_ok=True, symlinks=False)
        if not restore_valid_bin(bin_dir, env):
            raise RuntimeError("Built runtime did not enumerate the G4 CUDA device")
        local_tar = work / "runtime.tar"
        with tarfile.open(local_tar, "w", dereference=True) as bundle:
            bundle.add(bin_dir, arcname="bin")
        checksum_value = digest_file(local_tar)
        partial = cache / f"runtime.tar.{SERVICE}.{os.getpid()}.partial"
        shutil.copyfile(local_tar, partial)
        partial.replace(archive)
        checksum.write_text(checksum_value + "\n")
        write_json_atomic(cache / "manifest.json", signature)
        local_tar.unlink()
        qlog("Complete runtime cached on Drive: " + str(cache))
    help_text = cmd_out([str(bin_dir / "llama-server"), "--help"], env=env)
    for flag in ("--lazy-mode", "--load-mode", "--api-key-file"):
        if flag not in help_text:
            raise RuntimeError("Pinned llama.cpp lacks required option: " + flag)
    write_json_atomic(local_manifest, signature)
    return bin_dir, env

def hf_download_client(work):
    os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
    try:
        import hf_xet
        from huggingface_hub import HfApi, hf_hub_download
    except ImportError:
        run_logged([sys.executable, "-m", "pip", "install", "-U", "huggingface_hub", "hf_xet"], work / "download.log")
        from huggingface_hub import HfApi, hf_hub_download
    return HfApi, hf_hub_download

def valid_shard(path, item):
    return path.is_file() and path.stat().st_size == item["size"] and digest_file(path) == item["sha256"]

def prepare_shard(item, config, revision, local_dir, download):
    target = local_dir / item["name"]
    target.parent.mkdir(parents=True, exist_ok=True)
    if download is not None:
        try:
            qlog("Downloading HF -> local SSD (resumable): " + item["name"])
            downloaded = Path(download(repo_id=config["repo"], filename=item["name"],
                                       revision=revision, local_dir=local_dir / ".hf-download"))
            if not valid_shard(downloaded, item):
                downloaded.unlink(missing_ok=True)
                raise RuntimeError("HF shard size/SHA256 mismatch")
            downloaded.replace(target)
            qlog("SHA256 OK (HF): " + target.name)
            return target
        except Exception as error:
            qlog(f"HF download failed ({type(error).__name__}); trying Drive backup: " + item["name"])

    source = Path(config["model_root"]) / config["repo"] / item["name"]
    if not source.is_file() or source.stat().st_size != item["size"]:
        raise RuntimeError("HF download unavailable and no complete Drive backup: " + item["name"])
    qlog("Copying Drive backup -> local SSD and verifying SHA256: " + item["name"])
    partial = target.with_suffix(".gguf.drive.partial")
    h = hashlib.sha256()
    copied, last_report = 0, time.monotonic()
    with source.open("rb") as src, partial.open("wb") as dst:
        while chunk := src.read(16 * 1024 * 1024):
            dst.write(chunk)
            h.update(chunk)
            copied += len(chunk)
            if time.monotonic() - last_report >= 30:
                qlog(f"Copy {target.name}: {copied/1e9:.1f}/{item['size']/1e9:.1f} GB")
                last_report = time.monotonic()
    if copied != item["size"] or h.hexdigest() != item["sha256"]:
        partial.unlink(missing_ok=True)
        raise RuntimeError("Drive backup size/SHA256 mismatch: " + item["name"])
    partial.replace(target)
    qlog("SHA256 OK (Drive): " + target.name)
    return target

def prepare_model(config, service, work):
    download = None
    lock_path = service / ("model-" + config["quant"] + ".json")
    if lock_path.exists():
        lock = json.loads(lock_path.read_text())
    elif config["repo"] == DEFAULT_MODEL_LOCK["repo"] and config["quant"] == "IQ3_S":
        lock = DEFAULT_MODEL_LOCK
        write_json_atomic(lock_path, lock)
    else:
        HfApi, download = hf_download_client(work)
        info = HfApi().model_info(config["repo"], files_metadata=True)
        files = []
        for entry in info.siblings:
            if entry.rfilename.startswith(config["quant"] + "/") and entry.rfilename.endswith(".gguf"):
                if not entry.lfs:
                    raise RuntimeError("Expected LFS SHA256 metadata")
                files.append({"name": entry.rfilename, "size": entry.size, "sha256": entry.lfs.sha256})
        if len(files) != 2:
            raise RuntimeError(f"Expected two GGUF shards, got {len(files)}")
        lock = {"repo": config["repo"], "revision": info.sha, "files": sorted(files, key=lambda x: x["name"])}
        write_json_atomic(lock_path, lock)
    if lock["repo"] != config["repo"]:
        raise RuntimeError("Model lock belongs to a different repository")
    local_dir = work / "model"
    local_dir.mkdir(exist_ok=True)
    marker = local_dir / "verified.json"
    expected = {item["name"]: item["sha256"] for item in lock["files"]}
    if marker.exists() and json.loads(marker.read_text()) == expected and all(
        (local_dir / item["name"]).is_file() and (local_dir / item["name"]).stat().st_size == item["size"]
        for item in lock["files"]
    ):
        qlog("Reusing previously verified local model shards")
        return local_dir / lock["files"][0]["name"]
    pending = [item for item in lock["files"] if not valid_shard(local_dir / item["name"], item)]
    if pending:
        require_space(work, sum(item["size"] for item in pending) + 15 * 1024**3)
        if download is None:
            try:
                _, download = hf_download_client(work)
            except Exception as error:
                qlog(f"HF client unavailable ({type(error).__name__}); using Drive backups")
        with ThreadPoolExecutor(max_workers=2) as pool:
            futures = [pool.submit(prepare_shard, item, config, lock["revision"], local_dir, download)
                       for item in pending]
            for future in futures:
                future.result()
    write_json_atomic(marker, expected)
    return local_dir / lock["files"][0]["name"]

def prepare_projector(config, service, work):
    item = config.get("vision_projector")
    if not item:
        return None
    lock = json.loads((service / ("model-" + config["quant"] + ".json")).read_text())
    if lock["repo"] != config["repo"] or lock["revision"] != item["revision"]:
        raise RuntimeError("Vision projector must match the pinned model repository/revision")
    local_dir = work / "model"
    local_dir.mkdir(exist_ok=True)
    target = local_dir / item["name"]
    if valid_shard(target, item):
        qlog("Reusing verified local vision projector")
        return target
    require_space(work, item["size"] + 1024**3)
    try:
        _, download = hf_download_client(work)
    except Exception as error:
        qlog(f"HF client unavailable ({type(error).__name__}); using Drive projector backup")
        download = None
    target = prepare_shard(item, config, item["revision"], local_dir, download)
    backup = Path(config["model_root"]) / config["repo"] / item["name"]
    if not valid_shard(backup, item):
        backup.parent.mkdir(parents=True, exist_ok=True)
        partial = backup.with_suffix(f".gguf.{SERVICE}.{os.getpid()}.partial")
        shutil.copyfile(target, partial)
        if not valid_shard(partial, item):
            partial.unlink(missing_ok=True)
            raise RuntimeError("Vision projector backup size/SHA256 mismatch")
        partial.replace(backup)
    return target

if MOUNT_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive is not mounted. Run via `colabllm qwen-flash up`.")
if not MOUNT_DRIVE:   # --no-drive: keep durable state/model backups on local SSD
    for _key, _sub in (("service_dir", "service"), ("model_root", "models")):
        if CFG[_key].startswith("/content/drive/"):
            CFG[_key] = str(Path(DATA_DIR) / _sub)
SERVICE_DIR = Path(CFG["service_dir"])
SERVICE_DIR.mkdir(parents=True, exist_ok=True)
QWEN_WORK = Path(CFG["work_dir"])
QWEN_WORK.mkdir(parents=True, exist_ok=True)

qlog("Preparing model and runtime concurrently; durable state: " + str(SERVICE_DIR))
with ThreadPoolExecutor(max_workers=2) as pool:
    f_build = pool.submit(prepare_runtime, CFG, SERVICE_DIR, QWEN_WORK)
    f_model = pool.submit(prepare_model, CFG, SERVICE_DIR, QWEN_WORK)
    while not (f_build.done() and f_model.done()):
        for fut in (f_build, f_model):
            if fut.done() and fut.exception():
                raise fut.exception()
        qlog(f"Preparation heartbeat | build={'done' if f_build.done() else 'running'} "
             f"model={'done' if f_model.done() else 'running'}")
        time.sleep(30)
    BIN_DIR, LLAMA_ENV = f_build.result()
    MODEL_PATH = f_model.result()
PROJECTOR_PATH = prepare_projector(CFG, SERVICE_DIR, QWEN_WORK)
qlog(f"Setup complete: model={MODEL_PATH} projector={PROJECTOR_PATH}")


In [ ]:
# @title 2. Service -- llama-server on 127.0.0.1 (--load-mode none --lazy-mode off, YaRN 800K, vision, --api-key-file)
KEY_FILE = QWEN_WORK / "api_key.txt"
KEY_FILE.write_text(API_TOKEN + "\n")
KEY_FILE.chmod(0o600)

SERVER_ARGS = [
    str(BIN_DIR / "llama-server"), "-m", str(MODEL_PATH), "--alias", CFG["alias"],
    "--host", "127.0.0.1", "--port", str(SERVICE_PORT), "-ngl", "all",
    "-c", str(CFG["context"]), "-np", "1", "-lm", "none", "--lazy-mode", "off",
    "-ctk", CFG["cache_type"], "-ctv", CFG["cache_type"], "-fa", "on",
    "-b", str(CFG["batch"]), "-ub", str(CFG["ubatch"]),
    "--jinja", "--api-key-file", str(KEY_FILE), "--no-context-shift", "-lv", "4",
]
if PROJECTOR_PATH is not None:
    SERVER_ARGS += ["--mmproj", str(PROJECTOR_PATH)]
if CFG.get("rope_scaling"):
    SERVER_ARGS += [
        "--rope-scaling", CFG["rope_scaling"],
        "--rope-scale", str(CFG["rope_scale"]),
        "--yarn-orig-ctx", str(CFG["yarn_orig_ctx"]),
    ]

def start_service():
    subprocess.run(["pkill", "-f", "llama-server"], check=False)
    qlog(f"Starting llama-server: ctx={CFG['context']}, KV={CFG['cache_type']}")
    return ct_spawn("llama-server", SERVER_ARGS, env=LLAMA_ENV)

PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- verify local/public auth, 800K context, vision, smoke test, then supervise
BROWSER_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"

def http_json(url, key=None, payload=None, timeout=15):
    headers = {"Content-Type": "application/json", "User-Agent": BROWSER_UA}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    req = urllib.request.Request(url, headers=headers,
                                 data=None if payload is None else json.dumps(payload).encode())
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, json.loads(resp.read())
    except urllib.error.HTTPError as err:
        return err.code, {}

def auth_check(base, key):
    for candidate, expected in ((None, 401), ("invalid-deployment-test-key", 401), (key, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Authentication test failed on {base}: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
deadline = time.monotonic() + 1800
last_notice = 0
while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        raise RuntimeError("llama-server exited during load:\n" + ct_tail("llama-server", 40))
    try:
        if http_json(LOCAL_BASE + "/health", API_TOKEN)[0] == 200:
            break
    except (OSError, ValueError):
        pass
    if time.monotonic() - last_notice > 60:
        qlog("Waiting for model load / warmup | " + ct_gpu_stat("memory.used,utilization.gpu"))
        last_notice = time.monotonic()
    time.sleep(5)
else:
    raise RuntimeError("Model load timeout")

auth_check(LOCAL_BASE, API_TOKEN)
code, props = http_json(LOCAL_BASE + "/props", API_TOKEN)
allocated = props.get("default_generation_settings", {}).get("n_ctx")
if code != 200 or allocated != CFG["context"]:
    raise RuntimeError(f"Actual context differs from requested context: {allocated}")
vision_enabled = props.get("modalities", {}).get("vision", False)
if PROJECTOR_PATH is not None and not vision_enabled:
    raise RuntimeError("Vision projector configured but server does not advertise vision")
qlog(f"Local verified: auth=401/401/200, n_ctx={allocated}, vision={vision_enabled}")

# Public endpoint verification (DNS/edge propagation can take a few seconds)
ct_public_gate(PUBLIC_URL, lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
               lambda: auth_check(PUBLIC_URL, API_TOKEN), qlog, timeout=120, interval=10)
code, resp = http_json(PUBLIC_URL + "/v1/chat/completions", API_TOKEN, {
    "model": CFG["alias"],
    "messages": [{"role": "user", "content": "Reply with exactly two words: deployment successful"}],
    "max_tokens": 64, "temperature": 0,
    "chat_template_kwargs": {"enable_thinking": False},
}, timeout=180)
if code != 200:
    raise RuntimeError(f"Public smoke completion failed: HTTP {code}")
reply = (resp.get("choices") or [{}])[0].get("message", {}).get("content", "")
qlog(f"PUBLIC VERIFIED: {PUBLIC_URL}/v1 | reply={reply[:60]}")
ver_rec = {
    "service": SERVICE,
    "verified_at": time.strftime("%FT%T%z"), "base_url": PUBLIC_URL + "/v1",
    "context_allocated": allocated, "vision_enabled": vision_enabled, "reply": reply,
}
write_json_atomic(SERVICE_DIR / "last-verification.json", ver_rec)
if SERVICE != "qwen-flash":
    write_json_atomic(SERVICE_DIR / f"last-verification-{SERVICE}.json", ver_rec)

print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

def qwen_heartbeat(uptime):
    gpu_mem = ct_gpu_stat("memory.used,memory.total")
    status_rec = {
        "service": SERVICE,
        "kernel_pid": os.getpid(), "server_pid": PROCS["service"][0].pid,
        "tunnel_pid": PROCS[TUNNEL_PROCESS_KEY][0].pid, "context": CFG["context"],
        "alias": CFG["alias"], "url": PUBLIC_URL + "/v1",
        "updated_at": time.strftime("%FT%T%z"), "uptime_sec": uptime,
        "gpu_memory": gpu_mem,
    }
    try:
        write_json_atomic(SERVICE_DIR / "last-status.json", status_rec)
        if SERVICE != "qwen-flash":
            write_json_atomic(SERVICE_DIR / f"last-status-{SERVICE}.json", status_rec)
    except OSError:
        pass
    qlog("heartbeat | " + gpu_mem)

ct_watchdog(lambda: http_json(LOCAL_BASE + "/health", API_TOKEN)[0] == 200, start_service, qlog,
            log_name="llama-server", heartbeat=qwen_heartbeat)
